# 04_1. 장애인·노인 편의 접근성 재설계

- 목적: 기존 도보 400m 단독 한계를 보완하되, 일반 문화시설 접근성의 중분류별 이동수단 기준과 일관되게 장애인·노인 편의 접근성을 재산출함.
- 기존 `04_disability_elderly_e2sfca.ipynb`는 수정하지 않음.
- 산출물 저장은 기본값 `SAVE_OUTPUTS = False`로 잠가둠.

## 1. 분석 설계

- 도보형 중분류: 도서, 문화체험, 음악, 영상, 체육시설, 체육용품.
- 대중교통형 중분류: 공연, 미술, 스포츠관람, 관광지.
- 장애인친화시설은 `장애인친화시설 == 1`인 가맹점을 사용함.
- 노인편의서비스는 `전화결제 == 1` 또는 `찾아가는문화서비스 == 1`인 가맹점을 사용함.
- 접근성 계산은 중분류별 결과로 나누지 않고, 지표별 가맹점 단위로 통합함.
- 다만 대시보드 원인 설명을 위해 도달 가능 중분류와 부족 중분류는 별도 진단값으로 보존함.
- 공급량은 분류별 보정공급량을 쓰지 않고, `가맹점 1개 = 공급 1`로 처리함.

## 2. 거리감쇠 기준

- 도보형 중분류: `0~400m = 1.00`, `400~750m = 0.50`, `750m 초과 = 0`.
- 대중교통형 중분류: 총 접근시간 기준 `0~10분 = 1.00`, `10~15분 = 0.50`, `15~20분 = 0.20`, `20분 초과 = 0`.
- 대중교통 총 접근시간에는 격자-정류장 보행시간, 대중교통 네트워크 시간, 하차 후 가맹점 보행시간이 이미 포함되어 있음.
- 일반 접근성 분석과 일관성을 유지하기 위해 출발/도착 보행감쇠를 별도로 곱하지 않음.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", "{:,.6f}".format)

BASE_PATH = Path().resolve()

if BASE_PATH.name == "access":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "analysis_table").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

ANALYSIS_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output"
NETWORK_OUTPUT_PATH = ANALYSIS_OUTPUT_PATH / "network_competition_25km"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "disability_elderly_accessibility_mode_consistent"
DOCS_PATH = PROJECT_PATH / "notebooks" / "access" / "docs"

GRID_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet"
STORE_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet"
WALK_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
TRANSIT_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"
SEOUL_MNC_DISABLED_PATH = ANALYSIS_OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
EXTERNAL_AGE_PATH = ANALYSIS_OUTPUT_PATH / "인천경기_외부25km_100m_성연령별_추정인구.parquet"
DISABLED_SIGUNGU_PATH = PROJECT_PATH / "data" / "grid" / "시군구별_장애정도별_성별_등록장애인수_20260814052825.csv"
RESIDENT_POP_PATH = PROJECT_PATH / "data" / "grid" / "202410_202410_주민등록인구및세대현황_월간.csv"

WALK_CATEGORIES = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
TRANSIT_CATEGORIES = ["공연", "미술", "스포츠관람", "관광지"]
ALL_ANALYSIS_CATEGORIES = WALK_CATEGORIES + TRANSIT_CATEGORIES

WALK_WEIGHT = 0.7
TRANSIT_WEIGHT = 0.3
SAVE_OUTPUTS = True

print("PROJECT_PATH:", PROJECT_PATH)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


## 3. 함수 정의

- 기존 수요 산정 방식은 유지함.
- 서울은 성·연령·장애 추정 인구를 직접 사용함.
- 인천·경기 외부 25km 수요는 시군구 장애인비율과 연령대 정보를 이용해 보정함.
- 접근성 계산에서는 같은 지표·같은 이동수단 안에서 동일 가맹점을 중복 계산하지 않음.

In [ ]:
def log(message):
    print(f"[장애인·노인 이동수단 일관 E2SFCA] {message}")


def clean_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def normalize_age(value):
    value = str(value)
    if value == "100세이상":
        return "100세-"
    return value


def join_unique(values):
    items = sorted({str(value).strip() for value in values if pd.notna(value) and str(value).strip()})
    return ", ".join(items)


def missing_categories(reached_text):
    reached = {item.strip() for item in str(reached_text).split(",") if item.strip()}
    missing = [category for category in ALL_ANALYSIS_CATEGORIES if category not in reached]
    return ", ".join(missing)


def parse_sigungu_population():
    pop = pd.read_csv(RESIDENT_POP_PATH, encoding="cp949")
    pop["총인구수"] = clean_number(pop["2024년10월_총인구수"]).fillna(0)
    parsed = pop["행정구역"].astype(str).str.extract(r"^(?P<행정명>.+?)\s*\((?P<행정코드>\d+)\)$")
    pop["행정명"] = parsed["행정명"].str.strip()
    pop["행정코드"] = parsed["행정코드"]
    sigungu = pop[pop["행정코드"].str.endswith("00000", na=False)].copy()
    sigungu = sigungu[~sigungu["행정코드"].str.endswith("00000000", na=False)].copy()
    parts = sigungu["행정명"].str.split()
    sigungu["시도"] = parts.str[0]
    sigungu["시군구"] = parts.apply(lambda x: " ".join(x[1:]) if isinstance(x, list) and len(x) > 1 else np.nan)
    return sigungu[["시도", "시군구", "총인구수"]]


def parse_disabled_ratio():
    disabled = pd.read_csv(DISABLED_SIGUNGU_PATH, encoding="utf-8-sig")
    disabled = disabled.iloc[2:].copy()
    disabled = disabled.rename(columns={
        disabled.columns[0]: "시도",
        disabled.columns[1]: "시군구",
        disabled.columns[2]: "등록장애인수",
    })
    disabled["등록장애인수"] = clean_number(disabled["등록장애인수"]).fillna(0)
    disabled = disabled[disabled["시도"].isin(["서울특별시", "경기도", "인천광역시"])].copy()
    disabled = disabled[disabled["시군구"].ne("소계")].copy()
    pop = parse_sigungu_population()
    ratio = disabled[["시도", "시군구", "등록장애인수"]].merge(pop, on=["시도", "시군구"], how="left")
    ratio["장애인비율"] = np.where(ratio["총인구수"] > 0, ratio["등록장애인수"] / ratio["총인구수"], np.nan)
    sido_ratio = ratio.groupby("시도", as_index=False)["장애인비율"].mean().rename(columns={"장애인비율": "시도_장애인비율"})
    ratio = ratio.merge(sido_ratio, on="시도", how="left")
    ratio["장애인비율"] = ratio["장애인비율"].fillna(ratio["시도_장애인비율"]).fillna(0).clip(0, 1)
    return ratio[["시도", "시군구", "장애인비율"]]


def load_grid():
    grid = pd.read_parquet(GRID_COMPETITION_PATH).drop(columns=["geometry"], errors="ignore")
    grid["서울여부"] = grid["서울여부"].astype(bool)
    return grid


def load_store():
    store = pd.read_parquet(STORE_COMPETITION_PATH).drop(columns=["geometry"], errors="ignore")
    for col in ["전화결제", "장애인친화시설", "찾아가는문화서비스"]:
        store[col] = pd.to_numeric(store[col], errors="coerce").fillna(0).astype(int)
    store["노령인구_편의서비스"] = np.where(
        store["전화결제"].eq(1) | store["찾아가는문화서비스"].eq(1),
        1,
        0,
    )
    store["공급량"] = 1.0
    return store


def build_grid_demand(grid):
    seoul = pd.read_csv(
        SEOUL_MNC_DISABLED_PATH,
        usecols=[
            "GRID_CD", "시군구", "행정동", "연령대",
            "문화누리대상자_성연령별_추정_인구수",
            "문화누리대상자_성연령장애별_추정_인구수",
        ],
        encoding="utf-8-sig",
    )
    seoul["연령대"] = seoul["연령대"].map(normalize_age)
    elderly_ages = ["70-79세", "80-89세", "90-99세", "100세-"]
    seoul["노령인구_수요인구수"] = np.where(
        seoul["연령대"].isin(elderly_ages),
        pd.to_numeric(seoul["문화누리대상자_성연령별_추정_인구수"], errors="coerce").fillna(0),
        0,
    )
    seoul_demand = seoul.groupby(["GRID_CD", "시군구", "행정동"], as_index=False, observed=True).agg(
        총_문화누리대상자=("문화누리대상자_성연령별_추정_인구수", "sum"),
        장애인_수요인구수=("문화누리대상자_성연령장애별_추정_인구수", "sum"),
        노령인구_수요인구수=("노령인구_수요인구수", "sum"),
    )

    disabled_ratio = parse_disabled_ratio()
    external = pd.read_parquet(EXTERNAL_AGE_PATH)
    for col in ["GRID_CD", "시도", "시군구", "행정동", "연령대"]:
        if col in external.columns:
            external[col] = external[col].astype(str)
    external["연령대"] = external["연령대"].map(normalize_age)
    external = external.merge(disabled_ratio, on=["시도", "시군구"], how="left")
    sido_ratio = disabled_ratio.groupby("시도", as_index=False)["장애인비율"].mean().rename(columns={"장애인비율": "시도_장애인비율"})
    external = external.merge(sido_ratio, on="시도", how="left")
    external["장애인비율"] = external["장애인비율"].fillna(external["시도_장애인비율"]).fillna(0).clip(0, 1)
    external["총_문화누리대상자"] = pd.to_numeric(external["문화누리대상자_성연령별_추정_인구수"], errors="coerce").fillna(0)
    external["장애인_수요인구수"] = external["총_문화누리대상자"] * external["장애인비율"]
    external["노령인구_수요인구수"] = np.where(external["연령대"].isin(elderly_ages), external["총_문화누리대상자"], 0)
    external_demand = external.groupby(["GRID_CD", "시군구", "행정동"], as_index=False, observed=True).agg(
        총_문화누리대상자=("총_문화누리대상자", "sum"),
        장애인_수요인구수=("장애인_수요인구수", "sum"),
        노령인구_수요인구수=("노령인구_수요인구수", "sum"),
    )
    demand = pd.concat([seoul_demand, external_demand], ignore_index=True)
    demand = demand.groupby(["GRID_CD", "시군구", "행정동"], as_index=False, observed=True).sum()
    for col in ["총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수"]:
        demand[col] = pd.to_numeric(demand[col], errors="coerce").fillna(0).clip(lower=0)
    log(f"수요 테이블: {demand.shape}")
    return demand


def walk_decay(cost):
    cost = pd.to_numeric(cost, errors="coerce")
    weight = np.zeros(len(cost), dtype="float32")
    weight[(cost >= 0) & (cost <= 400)] = 1.00
    weight[(cost > 400) & (cost <= 750)] = 0.50
    return weight


def transit_decay(cost):
    cost = pd.to_numeric(cost, errors="coerce")
    weight = np.zeros(len(cost), dtype="float32")
    weight[(cost >= 0) & (cost <= 10)] = 1.00
    weight[(cost > 10) & (cost <= 15)] = 0.50
    weight[(cost > 15) & (cost <= 20)] = 0.20
    return weight


def positive_rank_score(series, mask):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    score = pd.Series(0.0, index=series.index, dtype="float64")
    positive_mask = mask & values.gt(0)
    if positive_mask.sum() > 1:
        score.loc[positive_mask] = values.loc[positive_mask].rank(pct=True, method="average") * 100
    elif positive_mask.sum() == 1:
        score.loc[positive_mask] = 100.0
    return score


## 4. 접근 pair 구성

- 도보 pair는 도보형 중분류 가맹점만 사용함.
- 대중교통 pair는 대중교통형 중분류 가맹점만 사용함.
- 장애인·노인 편의 플래그가 없는 가맹점은 계산 전 제거해 메모리 사용을 줄임.

In [ ]:
def load_mode_pair(store, mode):
    if mode == "도보":
        path = WALK_PAIR_PATH
        categories = WALK_CATEGORIES
        decay_func = walk_decay
    elif mode == "대중교통":
        path = TRANSIT_PAIR_PATH
        categories = TRANSIT_CATEGORIES
        decay_func = transit_decay
    else:
        raise ValueError(f"알 수 없는 접근수단입니다: {mode}")

    service_store = store[
        store["중분류"].isin(categories)
        & (store["장애인친화시설"].eq(1) | store["노령인구_편의서비스"].eq(1))
    ].copy()
    facility_cols = [
        "가맹점_ID", "가맹점명", "중분류", "소분류",
        "전화결제", "장애인친화시설", "찾아가는문화서비스", "노령인구_편의서비스", "공급량"
    ]
    pair = pd.read_parquet(path, columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"])
    pair["접근비용"] = pd.to_numeric(pair["접근비용"], errors="coerce").astype("float32")
    pair = pair.merge(service_store[facility_cols], on="가맹점_ID", how="inner")
    pair["접근수단"] = mode
    pair["거리감쇠"] = decay_func(pair["접근비용"])
    pair = pair[pair["거리감쇠"] > 0].copy()
    log(f"{mode} 편의시설 접근 pair: {pair.shape}, GRID {pair['GRID_CD'].nunique():,}, STORE {pair['가맹점_ID'].nunique():,}")
    return pair


def build_access_pair(store):
    walk_pair = load_mode_pair(store, "도보")
    transit_pair = load_mode_pair(store, "대중교통")
    pair = pd.concat([walk_pair, transit_pair], ignore_index=True)
    log(f"도보+대중교통 통합 pair: {pair.shape}, GRID {pair['GRID_CD'].nunique():,}, STORE {pair['가맹점_ID'].nunique():,}")
    return pair


## 5. E2SFCA 계산

- 지표별·이동수단별로 E2SFCA를 계산함.
- 같은 이동수단 안에서 동일 격자-가맹점이 여러 중분류로 반복되면 가장 강한 거리감쇠 경로만 사용함.
- 최종 통합접근성은 도보형 접근성 점수 0.7, 대중교통형 접근성 점수 0.3으로 결합함.

In [ ]:
def calculate_mode_e2sfca(grid, pair, demand, indicator_name, flag_col, demand_col, mode):
    target = pair[(pair[flag_col].eq(1)) & (pair["접근수단"].eq(mode))].copy()
    if len(target) == 0:
        access = grid.loc[grid["서울여부"], ["GRID_CD"]].copy()
        access[f"{indicator_name}_{mode}_E2SFCA"] = 0.0
        access[f"{indicator_name}_{mode}_도달가맹점수"] = 0
        access[f"{indicator_name}_{mode}_평균접근비용"] = 0.0
        return access, pd.DataFrame()

    seoul_ids = set(grid.loc[grid["서울여부"], "GRID_CD"])
    category_diag = (
        target[target["GRID_CD"].isin(seoul_ids)]
        .groupby("GRID_CD", as_index=False)
        .agg(**{f"{indicator_name}_{mode}_도달가능중분류": ("중분류", join_unique)})
    )

    target = target.sort_values(
        ["GRID_CD", "가맹점_ID", "거리감쇠", "접근비용"],
        ascending=[True, True, False, True],
    )
    target = target.drop_duplicates(["GRID_CD", "가맹점_ID"])
    target = target.merge(demand[["GRID_CD", demand_col]], on="GRID_CD", how="left")
    target[demand_col] = target[demand_col].fillna(0)
    target["가중수요"] = target[demand_col] * target["거리감쇠"]

    facility = target.groupby("가맹점_ID", as_index=False).agg(
        가맹점명=("가맹점명", "first"),
        대표중분류=("중분류", "first"),
        도달가능중분류=("중분류", join_unique),
        접근수단=("접근수단", "first"),
        공급량=("공급량", "first"),
        가중수요=("가중수요", "sum"),
    )
    facility["공급수요비"] = np.where(facility["가중수요"] > 0, facility["공급량"] / facility["가중수요"], 0)
    facility["지표"] = indicator_name

    calc = target.merge(facility[["가맹점_ID", "공급수요비"]], on="가맹점_ID", how="left")
    calc["접근성기여"] = calc["공급수요비"].fillna(0) * calc["거리감쇠"]
    access = calc[calc["GRID_CD"].isin(seoul_ids)].groupby("GRID_CD", as_index=False).agg(
        접근성지수=("접근성기여", "sum"),
        접근가능_가맹점수=("가맹점_ID", "nunique"),
        평균접근비용=("접근비용", "mean"),
    )
    access = access.rename(columns={
        "접근성지수": f"{indicator_name}_{mode}_E2SFCA",
        "접근가능_가맹점수": f"{indicator_name}_{mode}_도달가맹점수",
        "평균접근비용": f"{indicator_name}_{mode}_평균접근비용",
    })
    access = access.merge(category_diag, on="GRID_CD", how="left")
    return access, facility


def build_indicator_result(grid, pair, demand, indicator_name, flag_col, demand_col):
    walk_access, walk_facility = calculate_mode_e2sfca(grid, pair, demand, indicator_name, flag_col, demand_col, "도보")
    transit_access, transit_facility = calculate_mode_e2sfca(grid, pair, demand, indicator_name, flag_col, demand_col, "대중교통")

    seoul_base = grid[grid["서울여부"]][["GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수"]].copy()
    result = seoul_base.merge(
        demand[["GRID_CD", "총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수"]],
        on="GRID_CD",
        how="left",
    )
    result = result.merge(walk_access, on="GRID_CD", how="left")
    result = result.merge(transit_access, on="GRID_CD", how="left")

    numeric_cols = [
        f"{indicator_name}_도보_E2SFCA",
        f"{indicator_name}_대중교통_E2SFCA",
        f"{indicator_name}_도보_도달가맹점수",
        f"{indicator_name}_대중교통_도달가맹점수",
        f"{indicator_name}_도보_평균접근비용",
        f"{indicator_name}_대중교통_평균접근비용",
        "총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수",
    ]
    for col in numeric_cols:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(0)

    text_cols = [
        f"{indicator_name}_도보_도달가능중분류",
        f"{indicator_name}_대중교통_도달가능중분류",
    ]
    for col in text_cols:
        if col not in result.columns:
            result[col] = ""
        result[col] = result[col].fillna("")

    result[f"{indicator_name}_도달가능중분류"] = (
        result[f"{indicator_name}_도보_도달가능중분류"].astype(str)
        + ", "
        + result[f"{indicator_name}_대중교통_도달가능중분류"].astype(str)
    ).str.replace(r"^,\s*|,\s*$", "", regex=True)
    result[f"{indicator_name}_도달가능중분류"] = result[f"{indicator_name}_도달가능중분류"].apply(
        lambda value: join_unique(str(value).split(","))
    )
    result[f"{indicator_name}_부족중분류"] = result[f"{indicator_name}_도달가능중분류"].apply(missing_categories)

    analysis_mask = result["총_문화누리대상자"].fillna(0).gt(0)
    result[f"{indicator_name}_도보접근성점수"] = positive_rank_score(result[f"{indicator_name}_도보_E2SFCA"], analysis_mask)
    result[f"{indicator_name}_대중교통접근성점수"] = positive_rank_score(result[f"{indicator_name}_대중교통_E2SFCA"], analysis_mask)
    result[f"{indicator_name}_통합접근성"] = (
        WALK_WEIGHT * result[f"{indicator_name}_도보접근성점수"]
        + TRANSIT_WEIGHT * result[f"{indicator_name}_대중교통접근성점수"]
    )
    result[f"{indicator_name}_통합도달가맹점수"] = (
        result[f"{indicator_name}_도보_도달가맹점수"]
        + result[f"{indicator_name}_대중교통_도달가맹점수"]
    )

    facility = pd.concat([walk_facility, transit_facility], ignore_index=True)
    return result, facility


## 6. 실행 및 품질 점검

- GRID 중복, 좌표 결측, 수요 결측을 확인함.
- 접근성 0 비율을 기존 400m 단독 방식의 주요 문제와 비교하기 위한 핵심 점검값으로 확인함.
- 저장 스위치가 꺼져 있으면 CSV와 문서 메모를 생성하지 않음.

In [ ]:
def summarize_result(result):
    summary_rows = []
    for indicator, demand_col in [
        ("장애인친화시설", "장애인_수요인구수"),
        ("노인편의서비스", "노령인구_수요인구수"),
    ]:
        demand_mask = result[demand_col].fillna(0).gt(0)
        access_col = f"{indicator}_통합접근성"
        row = {
            "지표": indicator,
            "관련수요격자수": int(demand_mask.sum()),
            "관련수요합": float(result.loc[demand_mask, demand_col].sum()),
            "도보도달격자수": int(result[f"{indicator}_도보_도달가맹점수"].gt(0).sum()),
            "대중교통도달격자수": int(result[f"{indicator}_대중교통_도달가맹점수"].gt(0).sum()),
            "통합도달격자수": int(result[f"{indicator}_통합도달가맹점수"].gt(0).sum()),
            "관련수요_통합접근성0격자수": int((demand_mask & result[access_col].eq(0)).sum()),
            "관련수요_통합접근성0비율": float((demand_mask & result[access_col].eq(0)).sum() / max(demand_mask.sum(), 1)),
            "평균통합접근성": float(result.loc[demand_mask, access_col].mean()),
            "중앙통합접근성": float(result.loc[demand_mask, access_col].median()),
        }
        summary_rows.append(row)
    return pd.DataFrame(summary_rows)


def save_docs():
    text = """# 장애인·노인 편의 접근성 재설계

## 사용 데이터
- 서울 + 외부 25km 100m 격자 수요 테이블
- 서울 문화누리대상자 성연령장애 추정 인구
- 외부 25km 문화누리대상자 성연령 추정 인구
- 경기·인천·서울 시군구 등록장애인 비율
- 서울 + 외부 25km 문화누리 가맹점
- 도보 750m 격자-가맹점 pair
- 대중교통 20분 격자-가맹점 pair

## 전처리/분석 방식
- 기존 04번 도보 400m 단독 코드는 수정하지 않음.
- 장애인 지표 수요는 문화누리대상자 중 장애인 추정 인구를 사용함.
- 노인 지표 수요는 문화누리대상자 중 70세 이상 추정 인구를 사용함.
- 장애인친화시설은 장애인친화시설 flag가 1인 가맹점을 사용함.
- 노인 편의서비스는 전화결제 또는 찾아가는문화서비스 중 하나라도 제공하는 가맹점을 사용함.
- 도보형 중분류는 도보 750m, 대중교통형 중분류는 대중교통 20분 기준을 적용함.
- 도보 거리감쇠는 0~400m 1.00, 400~750m 0.50으로 적용함.
- 대중교통 거리감쇠는 총 접근시간 기준 0~10분 1.00, 10~15분 0.50, 15~20분 0.20으로 적용함.
- 일반 접근성 분석과 일관성을 유지하기 위해 대중교통에서 출발/도착 보행감쇠를 별도로 곱하지 않음.
- 접근성 공급량은 가맹점 1개를 1로 처리함.
- 최종 통합접근성은 도보 접근성 점수 0.7, 대중교통 접근성 점수 0.3으로 결합함.
- 접근성 점수화에서는 접근성 0 격자는 0점으로 두고, 양수 접근성 격자끼리만 순위점수를 부여함.

## 한계 및 보고 필요사항
- 대중교통망은 일반 대중교통 기준이며 엘리베이터, 저상버스, 환승 난이도, 보도 경사 등 장애인·고령자 이동 제약은 반영하지 못함.
- 온라인 가맹점은 공간 접근성 계산에 포함하지 않음.
"""
    DOCS_PATH.mkdir(parents=True, exist_ok=True)
    (DOCS_PATH / "disability_elderly_mode_consistent_e2sfca_전처리_사용데이터.txt").write_text(text, encoding="utf-8")


def save_outputs(result, facility, summary):
    ACCESS_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    result.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_중분류이동기준_E2SFCA_격자.csv", index=False, encoding="utf-8-sig")
    facility.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_중분류이동기준_E2SFCA_가맹점_공급수요비.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_중분류이동기준_E2SFCA_요약.csv", index=False, encoding="utf-8-sig")
    save_docs()
    log(f"산출물 저장 완료: {ACCESS_OUTPUT_PATH}")


def main():
    required_paths = [
        GRID_COMPETITION_PATH, STORE_COMPETITION_PATH, WALK_PAIR_PATH, TRANSIT_PAIR_PATH,
        SEOUL_MNC_DISABLED_PATH, EXTERNAL_AGE_PATH, DISABLED_SIGUNGU_PATH, RESIDENT_POP_PATH,
    ]
    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(path)

    grid = load_grid()
    store = load_store()
    demand = build_grid_demand(grid)

    quality = {
        "서울격자수": int(grid["서울여부"].sum()),
        "전체격자수": len(grid),
        "GRID중복수": int(grid["GRID_CD"].duplicated().sum()),
        "가맹점_ID_중복행수": int(store.duplicated(["가맹점_ID", "중분류", "소분류"]).sum()),
        "장애인친화_가맹점수": int(store.loc[store["장애인친화시설"].eq(1), "가맹점_ID"].nunique()),
        "노인편의_가맹점수": int(store.loc[store["노령인구_편의서비스"].eq(1), "가맹점_ID"].nunique()),
    }
    print("품질 점검")
    display(pd.DataFrame([quality]))

    pair = build_access_pair(store)
    disabled_result, disabled_facility = build_indicator_result(
        grid, pair, demand, "장애인친화시설", "장애인친화시설", "장애인_수요인구수"
    )
    elderly_result, elderly_facility = build_indicator_result(
        grid, pair, demand, "노인편의서비스", "노령인구_편의서비스", "노령인구_수요인구수"
    )

    result = disabled_result.merge(
        elderly_result[[
            "GRID_CD",
            "노인편의서비스_도보_E2SFCA", "노인편의서비스_대중교통_E2SFCA",
            "노인편의서비스_도보_도달가맹점수", "노인편의서비스_대중교통_도달가맹점수",
            "노인편의서비스_도보_평균접근비용", "노인편의서비스_대중교통_평균접근비용",
            "노인편의서비스_도보_도달가능중분류", "노인편의서비스_대중교통_도달가능중분류",
            "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
            "노인편의서비스_도보접근성점수", "노인편의서비스_대중교통접근성점수",
            "노인편의서비스_통합접근성", "노인편의서비스_통합도달가맹점수",
        ]],
        on="GRID_CD",
        how="left",
    )
    facility = pd.concat([disabled_facility, elderly_facility], ignore_index=True)
    summary = summarize_result(result)

    print("\n주요 결과 요약")
    display(summary)

    print("\n지표별 가맹점 수")
    if len(facility) > 0:
        display(facility.groupby(["지표", "접근수단"], as_index=False)["가맹점_ID"].nunique())

    print("\n중분류별 도달 가능 격자 수")
    category_rows = []
    for indicator in ["장애인친화시설", "노인편의서비스"]:
        for category in ALL_ANALYSIS_CATEGORIES:
            category_rows.append({
                "지표": indicator,
                "중분류": category,
                "도달가능격자수": int(result[f"{indicator}_도달가능중분류"].astype(str).str.contains(category, regex=False).sum()),
            })
    display(pd.DataFrame(category_rows))

    if SAVE_OUTPUTS:
        save_outputs(result, facility, summary)
    else:
        log("SAVE_OUTPUTS=False: CSV/문서 산출물을 생성하지 않았습니다.")

    return {"grid": result, "facility": facility, "summary": summary}


outputs = main()


## 7. 전처리 특이사항 및 판단

- 기존 400m 단독 접근성은 그대로 보존하고, 새 설계는 별도 노트북에서만 계산함.
- 대중교통 접근성은 일반 접근성 분석과 동일하게 총 접근시간에만 감쇠를 적용함.
- 장애인·노인 편의 플래그는 가맹점 수가 적으므로 분류별 접근성 점수로 나누지 않음.
- 부족 중분류는 점수 산출에는 넣지 않고, 대시보드 원인 설명용으로만 사용함.
- 접근성 0 격자는 점수화 단계에서 0점으로 고정하고, 양수 접근성 격자끼리만 순위점수를 부여함.